# CoordinatorAgent - Air Conditioner Remote Controller

This example demonstrates how to use the `CoordinatorAgent` with the DSPy subpackage to design an air conditioner remote controller. The coordinator decomposes the task into three specialized subtasks handled by different worker agents:

1. **Hardware Worker** - Electronic design (circuit board, IR transmitter, button matrix, display)
2. **Mechanical Worker** - Housing design (ergonomics, materials, button layout, battery compartment)
3. **Firmware Worker** - Programming (IR protocol encoding, button handling, power management)

The coordinator plans the steps, delegates to workers sequentially, and synthesizes a final design report.

In [ ]:
from typing import Dict, List, Sequence, Tuple

from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from aap_core.agent import BaseAgent
from aap_core.orchestration import CoordinatorAgent
from aap_core.types import AgentMessage, BaseLLMChain
from aap_dspy.chain import BaseSignatureAdapter, ChatCausalMultiTurnsChain

import dspy

## Setup Ollama Connection

We'll use `mistral:7b-instruct-v0.3-q4_K_S` (7.2B parameters) as our LLM backend.

In [2]:
# Connect to Ollama
model = dspy.LM(
    model="ollama_chat/mistral:7b-instruct-v0.3-q4_K_S",
    api_base="http://localhost:11434",
    api_key="",
    cache=False,
    max_tokens=4096
)

# State callback for monitoring execution
def state_callback(state: str):
    print(f"[Coordinator] {state}")

## Define Signature Adapters

We'll create a reusable signature adapter for worker agents that takes a query and returns a design response.

In [ ]:
class DesignSignature(dspy.Signature):
    """Signature for worker agents to produce design responses."""
    query: str = dspy.InputField()
    response: str = dspy.OutputField()
    history: dspy.History = dspy.InputField()


class DesignAdapter(BaseSignatureAdapter[DesignSignature]):
    """Adapter for converting between AgentMessage and DesignSignature."""

    _signature_map: Dict[str, str]
    _system_prompt: str = ""

    @classmethod
    def with_system_prompt(cls, system_prompt: str) -> "DesignAdapter":
        obj = cls()
        obj._system_prompt = system_prompt
        return obj

    def msg2sig(self, message: AgentMessage) -> List[DesignSignature]:
        history = dspy.History(messages=[])

        # Build history from previous responses
        q, a = None, None
        for response in message.responses:
            if response[0] == "user":
                q = response[1]
            elif response[0] not in ("tool", "system"):
                a = response[1]

            if q is not None and a is not None:
                history.messages.append({"question": q, "answer": a})
                q, a = None, None

        sig_dict = {
            "query": message.query,
            "response": "",
        }

        design_sig = DesignSignature(history=history, **sig_dict)
        return [design_sig]

    def sig2msg(self, signatures: List[DesignSignature], name: str) -> List:
        responses = []
        for signature in signatures:
            responses.append((name, signature.response))
        return responses

## Define the Hardware Worker Chain

The hardware worker focuses on electronic design: circuit board, IR transmitter, button matrix, and display.

In [4]:
hardware_system_prompt = """You are an expert electronic hardware engineer specializing in IR remote controller design.
Your task is to design the electronic hardware for an air conditioner remote controller.

Provide a comprehensive hardware design that includes:
1. **Microcontroller**: Recommend an appropriate MCU (e.g., ESP32, STM32, or dedicated IR remote chip) with justification
2. **IR Transmitter Circuit**: Specify IR LED, driver transistor, and resistor values for reliable IR signal transmission
3. **Button Matrix**: Design the button layout and scanning circuit (e.g., 4x4 matrix, membrane switch)
4. **Display**: Recommend display type (OLED, LCD, or LED indicators) and connection method
5. **Power Supply**: Specify battery type (e.g., 2x AAA), power management, and low-power sleep modes
6. **IR Protocol Support**: List supported AC protocols (e.g., NEC, RC5, Samsung, LG, Daikin, Panasonic)

Format your response with clear sections. Include specific part numbers, component values, and circuit descriptions.
"""

hardware_adapter = DesignAdapter.with_system_prompt(hardware_system_prompt)
hardware_predictor = dspy.ChainOfThought(DesignSignature)
hardware_chain = ChatCausalMultiTurnsChain(
    signature=DesignSignature,
    predictor=hardware_predictor,
    adapter=hardware_adapter,
).with_lm(model)
hardware_chain.final_response_as_context("hardware")

## Define the Mechanical Worker Chain

The mechanical worker focuses on housing design: ergonomics, materials, button layout, and battery compartment.

In [5]:
mechanical_system_prompt = """You are an expert mechanical engineer specializing in consumer product enclosure design.
Your task is to design the mechanical housing for an air conditioner remote controller.

Provide a comprehensive mechanical design that includes:
1. **Ergonomics**: Describe the form factor, dimensions, grip design, and weight considerations
2. **Material Selection**: Recommend materials (e.g., ABS plastic, TPU rubber grips) with justification for durability, cost, and feel
3. **Button Design**: Specify button type (membrane, dome, tactile), layout, and labeling strategy
4. **Battery Compartment**: Design for easy battery access, cover mechanism, and battery type
5. **IR Window**: Design the IR transmitter window/portal for optimal signal direction
6. **Durability**: Specify drop resistance, IP rating, and environmental considerations
7. **Aesthetics**: Color scheme, branding placement, and user interface elements

Format your response with clear sections. Include specific dimensions, material grades, and manufacturing considerations (injection molding, assembly).
"""

mechanical_adapter = DesignAdapter.with_system_prompt(mechanical_system_prompt)
mechanical_predictor = dspy.ChainOfThought(DesignSignature)
mechanical_chain = ChatCausalMultiTurnsChain(
    signature=DesignSignature,
    predictor=mechanical_predictor,
    adapter=mechanical_adapter,
).with_lm(model)
mechanical_chain.final_response_as_context("mechanical")

## Define the Firmware Worker Chain

The firmware worker focuses on programming: IR protocol encoding, button handling, and power management.

In [6]:
firmware_system_prompt = """You are an expert embedded firmware engineer specializing in IR remote controller development.
Your task is to design the firmware for an air conditioner remote controller.

Provide a comprehensive firmware design that includes:
1. **IR Protocol Implementation**: Code/pseudocode for encoding IR signals (NEC, RC5, or protocol-specific formats)
2. **Button Handling**: Debouncing logic, key scanning algorithm, and long-press detection
3. **Display Driver**: Code for driving the display (OLED, LCD) to show temperature, mode, and status
4. **Power Management**: Sleep modes, wake-up sources, and battery level monitoring
5. **AC Protocol Support**: List supported AC brands/models and their IR code structures
6. **User Interface**: Mode switching (cool/heat/fan/dry), temperature control, timer functions
7. **Learning Function**: Optional IR code learning capability for universal remote functionality

Format your response with clear sections. Include code snippets in C/C++ or MicroPython with comments.
"""

firmware_adapter = DesignAdapter.with_system_prompt(firmware_system_prompt)
firmware_predictor = dspy.ChainOfThought(DesignSignature)
firmware_chain = ChatCausalMultiTurnsChain(
    signature=DesignSignature,
    predictor=firmware_predictor,
    adapter=firmware_adapter,
).with_lm(model)
firmware_chain.final_response_as_context("firmware")

## Define Worker Agents

Create the three worker agents using the chains defined above. Each agent wraps a chain and executes it.

In [7]:
class WorkerAgent(BaseAgent):
    chain: BaseLLMChain

    def execute(self, message: AgentMessage, **kwargs) -> AgentMessage:
        self.state = "running"
        message = self.chain.invoke(message, **kwargs)
        message.execution_result = "success"
        message.origin = self.card.name
        self.state = "idle"
        return message

# Create worker agents
hardware_skill = AgentSkill(id='hardware-skill', name="hardware skill", description="Electronic hardware design for IR remote controllers", tags=['hardware', 'electronic'])
hardware_card = AgentCard(name="hardware_agent", description="Hardware design specialist for IR remotes", skills=[hardware_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
hardware_agent = WorkerAgent(chain=hardware_chain, card=hardware_card, state_change_callback=state_callback)

mechanical_skill = AgentSkill(id='mechanical-skill', name="mechanical skill", description="Mechanical housing design for consumer electronics", tags=['mechanical', 'housing'])
mechanical_card = AgentCard(name="mechanical_agent", description="Mechanical design specialist for enclosures", skills=[mechanical_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
mechanical_agent = WorkerAgent(chain=mechanical_chain, card=mechanical_card, state_change_callback=state_callback)

firmware_skill = AgentSkill(id='firmware-skill', name="firmware skill", description="Embedded firmware development for IR remote controllers", tags=['firmware', 'programming'])
firmware_card = AgentCard(name="firmware_agent", description="Firmware design specialist for embedded systems", skills=[firmware_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
firmware_agent = WorkerAgent(chain=firmware_chain, card=firmware_card, state_change_callback=state_callback)

workers = [hardware_agent, mechanical_agent, firmware_agent]

## Define the Planner Agent

The planner agent takes the main device query and generates a structured plan with steps, worker assignments, and dependencies.

In [ ]:
class PlanStep(dspy.Signature):
    """A single step in the coordinator plan."""
    worker: str = dspy.InputField()
    message: str = dspy.InputField()
    dependencies: str = dspy.InputField()

class PlanSignature(dspy.Signature):
    """Signature for the planner to produce a JSON plan."""
    query: str = dspy.InputField()
    plan: str = dspy.OutputField(desc="JSON array of plan steps")
    history: dspy.History = dspy.InputField()

class PlanAdapter(BaseSignatureAdapter[PlanSignature]):
    """Adapter for planner signature."""

    _system_prompt: str = ""

    @classmethod
    def with_system_prompt(cls, system_prompt: str) -> "PlanAdapter":
        obj = cls()
        obj._system_prompt = system_prompt
        return obj

    def msg2sig(self, message: AgentMessage) -> List[PlanSignature]:
        history = dspy.History(messages=[])

        q, a = None, None
        for response in message.responses:
            if response[0] == "user":
                q = response[1]
            elif response[0] not in ("tool", "system"):
                a = response[1]

            if q is not None and a is not None:
                history.messages.append({"question": q, "answer": a})
                q, a = None, None

        sig_dict = {
            "query": message.query,
            "plan": "",
        }

        plan_sig = PlanSignature(history=history, **sig_dict)
        return [plan_sig]

    def sig2msg(self, signatures: List[PlanSignature], name: str) -> List:
        responses = []
        for signature in signatures:
            responses.append((name, signature.plan))
        return responses

planner_system_prompt = """You are a project coordinator for designing an air conditioner remote controller.
Your job is to break down the device design task into clear, sequential steps and assign each step to the appropriate worker.

Available workers:
1. hardware_agent - Electronic hardware design (circuit, IR transmitter, buttons, display, power)
2. mechanical_agent - Mechanical housing design (ergonomics, materials, button layout, battery)
3. firmware_agent - Firmware programming (IR protocols, button handling, display driver, power management)

For each step, specify:
- worker: which worker handles this step (one of: hardware_agent, mechanical_agent, firmware_agent)
- message: a clear instruction for the worker about what to design
- dependencies: list of indices of previous steps this step depends on (empty list if no dependencies)

Rules:
- Always start with hardware design (step 0) since it informs mechanical and firmware decisions
- Mechanical design depends on hardware (needs to know component sizes and layout)
- Firmware design depends on hardware (needs to know which MCU and IR protocol is used)
- Each worker should receive the original query context plus any relevant dependencies

Output your plan as a JSON array of objects with keys: worker, message, dependencies.
Example format:
[
  {"worker": "hardware_agent", "message": "Design the electronic hardware...", "dependencies": []},
  {"worker": "mechanical_agent", "message": "Design the housing...", "dependencies": [0]},
  {"worker": "firmware_agent", "message": "Write the firmware...", "dependencies": [0]}
]

IMPORTANT: Return ONLY the JSON array, nothing else. No explanations, no markdown formatting."""

planner_adapter = PlanAdapter.with_system_prompt(planner_system_prompt)
planner_predictor = dspy.ChainOfThought(PlanSignature)
planner_chain = ChatCausalMultiTurnsChain(
    signature=PlanSignature,
    predictor=planner_predictor,
    adapter=planner_adapter,
).with_lm(model)
planner_chain.final_response_as_context("plan")

planner_skill = AgentSkill(id='planner-skill', name="planner skill", description="Task decomposition and planning", tags=['planner'])
planner_card = AgentCard(name="planner_agent", description="Project planning coordinator", skills=[planner_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
planner_agent = WorkerAgent(chain=planner_chain, card=planner_card, state_change_callback=state_callback)

## Implement Plan Parsing Function

This function parses the planner's JSON output into a sequence of tuples: (step_message, worker_agent, dependencies).

In [ ]:
import json
import re


def parse_plan(message: AgentMessage, workers: Sequence[BaseAgent]) -> Sequence[Tuple[AgentMessage, BaseAgent, List]]:
    """
    Parse the planner's output into a sequence of (step_message, worker_agent, dependencies) tuples.
    Handles both JSON and free-form text from the LLM.
    """
    # The planner chain uses final_response_as_context("plan"), so the response is in context["plan"]
    if message.context is None or "plan" not in message.context:
        # Fallback: try to get from responses if context is not set
        if message.responses:
            plan_text = message.responses[-1][1]
        else:
            raise ValueError("No plan found in message.context or message.responses")
    else:
        plan_text = str(message.context["plan"])

    # Map worker names to agent objects
    worker_map = {agent.card.name: agent for agent in workers}

    # Try 1: Extract JSON array from the response
    json_match = re.search(r'\[.*\]', plan_text, re.DOTALL)
    if json_match:
        try:
            plan = json.loads(json_match.group())
            steps = []
            for step in plan:
                worker_name = step["worker"]
                if worker_name not in worker_map:
                    raise ValueError(f"Unknown worker: {worker_name}. Available: {list(worker_map.keys())}")
                worker = worker_map[worker_name]
                dependencies = step.get("dependencies", [])
                step_msg = AgentMessage(query=step["message"])
                steps.append((step_msg, worker, dependencies))
            return steps
        except (json.JSONDecodeError, KeyError):
            pass  # Fall through to free-form parsing

    # Try 2: Parse free-form text by looking for worker assignments
    steps = []
    worker_names = list(worker_map.keys())
    lines = plan_text.split('\n')

    current_step = None
    step_index = 0

    for line in lines:
        line_lower = line.lower().strip()

        # Check if this line mentions a worker
        for worker_name in worker_names:
            if worker_name in line_lower:
                # Extract the message (everything after the worker name on this line or next few lines)
                msg_start = line_lower.find(worker_name) + len(worker_name)
                msg = line[msg_start:].strip().lstrip(':').strip()

                # If message is empty, look at next lines
                if not msg:
                    for next_line in lines[lines.index(line)+1:lines.index(line)+4]:
                        next_line = next_line.strip()
                        if next_line and not any(kw in next_line.lower() for kw in worker_names):
                            msg = next_line
                            break

                if msg:
                    # Try to extract dependencies from the line
                    deps_match = re.search(r'dependencies?\s*[:=]?\s*\[?(\d+(?:\s*,\s*\d+)*)\]?', line, re.IGNORECASE)
                    dependencies = []
                    if deps_match:
                        dependencies = [int(d.strip()) for d in deps_match.group(1).split(',') if d.strip().isdigit()]

                    step_msg = AgentMessage(query=msg)
                    steps.append((step_msg, worker_map[worker_name], dependencies))
                    step_index += 1
                    break

    if steps:
        return steps

    # Try 3: Fallback - create a single step for each worker with the full query
    steps = []
    for worker_name, worker in worker_map.items():
        step_msg = AgentMessage(query=f"Design the {worker_name} aspects of: {message.query}")
        steps.append((step_msg, worker, []))

    return steps

## Define Summary Agent for Final Synthesis

The summary agent aggregates results from all workers to produce a cohesive final design report.

In [10]:
summary_system_prompt = """You are a senior engineering manager reviewing an air conditioner remote controller design.
Your job is to synthesize the hardware, mechanical, and firmware design reports into a cohesive final design document.

You will receive:
- The original device query
- Hardware design results (context_results)
- Mechanical design results (context_results)
- Firmware design results (context_results)

Create a comprehensive final design report that:
1. Summarizes the key decisions from each domain
2. Identifies any conflicts or integration challenges between domains
3. Provides a unified implementation roadmap with phases
4. Lists all recommended components with part numbers
5. Includes a bill of materials (BOM) estimate
6. Suggests next steps for prototyping

Format the report with clear headings, tables for component lists, and actionable recommendations.
"""

summary_adapter = DesignAdapter.with_system_prompt(summary_system_prompt)
summary_predictor = dspy.ChainOfThought(DesignSignature)
summary_chain = ChatCausalMultiTurnsChain(
    signature=DesignSignature,
    predictor=summary_predictor,
    adapter=summary_adapter,
).with_lm(model)
summary_chain.final_response_as_context("summary")

summary_skill = AgentSkill(id='summary-skill', name="summary skill", description="Design synthesis and report generation", tags=['summary'])
summary_card = AgentCard(name="summary_agent", description="Design synthesis coordinator", skills=[summary_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")
summary_agent = WorkerAgent(chain=summary_chain, card=summary_card, state_change_callback=state_callback)

## Instantiate and Configure CoordinatorAgent

Combine the planner, workers, summary chain, and parsing function into a single CoordinatorAgent instance.

In [26]:
# Coordinator card
coordinator_skill = AgentSkill(id='coordinator-skill', name="coordinator skill", description="Orchestrates multi-agent device design", tags=['coordinator'])
coordinator_card = AgentCard(name="coordinator_agent", description="Multi-agent device design coordinator", skills=[coordinator_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url="localhost", version="0.1.0")

# Create the CoordinatorAgent
coordinator = CoordinatorAgent(
    planner_agent=planner_agent,
    parse_plan=parse_plan,
    workers=workers,
    summary_chain=summary_chain,
    summary_steps_key="context_results",
    card=coordinator_card,
    state_change_callback=state_callback,
)

## Execute the Coordinator

Now let's run the coordinator with a query about designing an air conditioner remote controller.

In [28]:
# Create the initial message
query = """Design an air conditioner remote controller with the following requirements:
- Support multiple AC brands (Daikin, LG, Samsung, Panasonic)
- OLED display showing temperature, mode, and timer
- 2x AAA battery power with low-power sleep mode
- Ergonomic design suitable for handheld use
- IR transmission range of at least 8 meters
- Budget-friendly manufacturing cost"""

message = AgentMessage(query=query)

# Execute the coordinator
result = coordinator.execute(message)

[Coordinator] coordinator_agent:planning/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:planning/((planner_agent:running)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:planning/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 0: worker hardware_agent running/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 0: worker hardware_agent running/((planner_agent:idle)-(hardware_agent:running)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 0: worker hardware_agent running/((planner_agent:idle)-(hardware_agent:idle)-(mechanical_agent:idle)-(firmware_agent:idle))
[Coordinator] coordinator_agent:step 1: worker mechanical_agent running/((planner_agent:idle)-(hardware_agent:idle)

## View the Results

Display the final synthesized design report from the coordinator.

In [44]:
# Display the final result
print("=== Coordinator Execution Complete ===")
print(f"Execution result: {result.execution_result}")
print(f"Origin: {result.origin}")
print(f"Context keys: {list(result.context.keys()) if result.context else 'None'}")

# The summary is stored in context, not in responses
if result.context and "summary" in result.context:
    print("\n" + "="*60)
    print("FINAL DESIGN REPORT")
    print("="*60)
    print(result.context["summary"])
    print("="*60)
else:
    print("\nNo summary found in context.")
    if result.error_message:
        print(f"Error: {result.error_message}")

=== Coordinator Execution Complete ===
Execution result: success
Origin: coordinator_agent
Context keys: ['results', 'summary']

FINAL DESIGN REPORT
Design for the air conditioner remote controller:
1. Universal IR code database to support multiple AC brands
2. OLED screen displaying temperature, mode, and timer
3. 2x AAA battery power with low-power sleep mode
4. Ergonomic design optimized for handheld use
5. Powerful IR transmitter with a minimum range of 8 meters
6. Budget-friendly manufacturing cost through the use of common components and cost-effective materials
